In [1]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots


os.chdir(os.path.dirname(os.getcwd()))

from src.constants.config import path_portfolio, path_db
from src.constants.tickers import isin_to_ticker, currencies

In [2]:
def search_database(path_db):
    """Returns Delisted stock from manual history db"""
    csv_files = [
        pd.read_csv(os.path.join(path_db, file), sep=";")
        for file in os.listdir(path_db)
        if file.endswith(".csv")
    ]
    df = pd.concat(csv_files, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y").dt.date

    return df


def collect_stock_rates(df):
    """Fetch historical stock data for a grouped DataFrame of tickers and return combined results."""
    all_data = []
    empty_data = []

    for _, row in df.iterrows():
        start_date = row["StartDate"]
        end_date = row["EndDate"]
        symbol = row["Ticker"]
        product = row["Product"]
        isin_code = row["ISIN"]

        try:
            ticker = yf.Ticker(symbol)
            currency = ticker.info["currency"]
            history_df = (
                ticker.history(start=start_date, end=end_date)
                .reset_index()[["Date", "Close"]]
                .assign(
                    Product=product, ISIN=isin_code, Ticker=symbol, currency=currency
                )
            )
            print(f"Retrieved {history_df.shape[0]} rows for {symbol}")
            all_data.append(history_df)

        except Exception as error:
            print(f"Error fetching data for {symbol}: {error}")
            empty_data.append(
                {
                    "Ticker": symbol,
                    "Product": product,
                    "ISIN": isin_code,
                    "StartDate": start_date,
                    "EndDate": end_date,
                    "Error": str(error),
                }
            )

    combined_df = pd.concat(all_data, ignore_index=True)
    empty_data = pd.DataFrame(empty_data)

    # read db
    print(f"loading delisted stocks from db:{empty_data['Ticker'].unique()}")
    df_db = search_database(path_db)
    df_db = df_db.loc[df_db["Ticker"].isin(empty_data["Ticker"].unique())]

    df = pd.concat([combined_df, df_db])
    df["Date"] = pd.to_datetime(df["Date"].astype(str).str.slice(0, 10)).dt.date

    return df


def collect_currency_data(currencies):
    """
    Fetch historical currency data using yfinance and return a combined DataFrame.

    Parameters:
        currencies (list): List of currency tickers.

    Returns:
        pd.DataFrame: Combined DataFrame with historical FX rates.
    """
    currency_data = []

    for currency in currencies:
        print(f"Fetching currency data for: {currency}")

        try:
            ticker = yf.Ticker(currency)
            cur_df = (
                ticker.history(period="max")
                .reset_index()[["Date", "Close"]]
                .assign(Product=currency)
                .assign(currency=currency.replace("=X", "").replace("EUR", ""))
                .rename(columns={"Close": "Fx_rate"})
            )

            print(f"Retrieved {cur_df.shape[0]} rows for {currency}")
            currency_data.append(cur_df)

        except Exception as error:
            print(f"Error fetching data for {currency}: {error}")

    combined_df = pd.concat(currency_data, ignore_index=True)
    combined_df["Date"] = combined_df["Date"].dt.date
    return combined_df

In [3]:
# load portfolio
df_portfolio = pd.read_csv(path_portfolio)

df_portfolio["Ticker"] = df_portfolio["ISIN"].replace(isin_to_ticker)
df_portfolio = df_portfolio.rename({"SnapshotDate": "Date"}, axis=1)
df_portfolio["Date"] = pd.to_datetime(df_portfolio["Date"].str.slice(0, 10))

df_grouped = df_portfolio.groupby(["Product", "ISIN", "Ticker"]).agg(
    {"Date": ["min", "max"]}
)
df_grouped.columns = ["StartDate", "EndDate"]
df_grouped = df_grouped.reset_index()

In [4]:
# Collect stocks
df_stock_rates = collect_stock_rates(df_grouped)

# Collect exchange rates
df_curr = collect_currency_data(currencies)

Retrieved 27 rows for ABN.AS
Retrieved 730 rows for BABA
Retrieved 1274 rows for AD.AS
Retrieved 868 rows for ASML.AS
Retrieved 186 rows for T
Retrieved 104 rows for BFIT.AS
Retrieved 1203 rows for BESI.AS
Retrieved 715 rows for FLOW.AS
Retrieved 312 rows for FLOW.AS
Retrieved 39 rows for GLPG.AS
Retrieved 15 rows for NL0000009165
Retrieved 28 rows for INGA.AS
Retrieved 1037 rows for IWDA.L
Retrieved 263 rows for IMAE.AS
Retrieved 92 rows for NFLX
Retrieved 773 rows for NKE


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ORDI"}}}


Error fetching data for ORDI: 'currency'
Retrieved 10 rows for PFE
Retrieved 46 rows for PHARM.AS
Retrieved 218 rows for PHIA.AS
Retrieved 1262 rows for PRX.AS
Retrieved 350 rows for SHELL.AS
Retrieved 418 rows for SHEL.L
Retrieved 1199 rows for TKWY.AS
Retrieved 75 rows for TKWY.AS
Retrieved 19 rows for ULVR.L
Retrieved 87 rows for IE00B945VV12
Retrieved 917 rows for VOW3.DE
Retrieved 11 rows for WCLD.L
loading delisted stocks from db:['ORDI']
Fetching currency data for: EURUSD=X
Retrieved 5689 rows for EURUSD=X
Fetching currency data for: GBPEUR=X
Retrieved 5742 rows for GBPEUR=X


In [5]:
df_portfolio["Date"] = df_portfolio["Date"].dt.date
df = df_portfolio.merge(
    df_stock_rates[["Date", "Ticker", "Close", "currency"]],
    on=["Date", "Ticker"],
    how="inner",
).merge(df_curr[["Date", "currency", "Fx_rate"]], on=["Date", "currency"], how="left")

df["Fx_rate"] = np.where(df["currency"] == "EUR", 1, df["Fx_rate"])
df["eur"] = df["Close"] / df["Fx_rate"] * df["Aantal"]

In [6]:
# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Create a Month-Year column
df['MonthYear'] = df['Date'].dt.to_period('M')

# Find last working day of each month
last_working_days = (
    df.groupby('MonthYear')['Date']
    .apply(lambda x: x[x.dt.is_month_end & ~x.dt.dayofweek.isin([5, 6])].max())
)

# Filter original dataframe to only include rows from last working day
df_filtered = df[df['Date'].isin(last_working_days.values)]

# Optional: drop MonthYear if you want to reassign it based on filtered dates
df_filtered['MonthYear'] = df_filtered['Date'].dt.to_period('M').astype(str)


/var/folders/c4/p_rq7mxn2knb64hrrwwdz4dc0000gn/T/ipykernel_20073/4292728839.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['MonthYear'] = df_filtered['Date'].dt.to_period('M').astype(str)


In [11]:
# Convert 'Date' to datetime and extract Month-Year
df['Date'] = pd.to_datetime(df['Date'])
df['MonthYear'] = df['Date'].dt.to_period('M')

# Find last calendar day of each month
last_days = df.groupby('MonthYear')['Date'].max()

# Filter to rows from last day of each month
df_filtered = df[df['Date'].isin(last_days.values)]

# Pivot for stacked bar chart
pivot_df = df_filtered.pivot_table(index='Date', columns='Product', values='eur', aggfunc='sum', fill_value=0)

# Create subplot figure
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.15,
    subplot_titles=['EUR per Product on Last Day of Month', 'Total EUR on Last Day of Month']
)

# Stacked bar chart per product
for product in pivot_df.columns:
    fig.add_trace(go.Bar(x=pivot_df.index, y=pivot_df[product], name=product), row=1, col=1)

# Total EUR per month
monthly_total = pivot_df.sum(axis=1)
fig.add_trace(go.Bar(x=monthly_total.index, y=monthly_total.values, name='Total EUR', marker_color='gray'), row=2, col=1)

# Layout settings
fig.update_layout(
    barmode='stack',
    height=700,
    xaxis_title='Month-Year',
    legend_title='Product',
    template='plotly_white'
)

fig.update_yaxes(title_text='EUR per Product', row=1, col=1)
fig.update_yaxes(title_text='Total EUR', row=2, col=1)

fig.show()
